<a href="https://colab.research.google.com/github/koushikkkkkkkkkk/prostate-mil-project/blob/Himesh/notebooks/03_abmil_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 - ABMIL Baseline Training on Real PANDA Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/koushikkkkkkkkkk/prostate-mil-project/blob/main/notebooks/03_abmil_training.ipynb)

**Role:** Person 1 (Team Leader & Baseline Owner)
**Task:** Train the gated-attention ABMIL baseline on real PANDA features across 3 seeds (0, 1, 2).

All datasets and checkpoints are stored directly in **Google Drive in the cloud** — zero large files are stored on your local laptop.


## 1. Setup & Environment
Clone the repository and install lightweight dependencies quietly without conflicting with Colab's pre-installed packages.


In [1]:
# Setup repository and dependencies quietly
import os, sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
    if not os.path.exists('prostate-mil-project'):
        !git clone https://github.com/koushikkkkkkkkkk/prostate-mil-project.git
        %cd prostate-mil-project
    else:
        %cd prostate-mil-project
        !git pull origin main
    !pip install -q pyyaml
except ImportError:
    IN_COLAB = False

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLBACKEND', 'Agg')

import torch
print(f'Running in Colab: {IN_COLAB}')
print(f'Working directory: {ROOT}')
print(f'PyTorch: {torch.__version__} | GPU Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')


Cloning into 'prostate-mil-project'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 147 (delta 56), reused 119 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 333.87 KiB | 8.56 MiB/s, done.
Resolving deltas: 100% (56/56), done.
/content/prostate-mil-project
Running in Colab: True
Working directory: /content/prostate-mil-project
PyTorch: 2.11.0+cu130 | GPU Available: True
GPU Device: Tesla T4


## 2. Mount Google Drive
All 420 slide feature files (~600 MB) and training outputs will stay safely in your Google Drive cloud storage.


In [ ]:
# Mount Google Drive (keeps all 600 MB of features safely in the cloud across sessions)
import shutil

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        GDRIVE_DIR = Path('/content/drive/MyDrive/prostate_mil_data')
        FEATURES_DIR = GDRIVE_DIR / 'features'
        BACKUP_DIR = GDRIVE_DIR / 'results'
        FEATURES_DIR.mkdir(parents=True, exist_ok=True)
        BACKUP_DIR.mkdir(parents=True, exist_ok=True)
        # Symlink data/features -> Google Drive so any command path works seamlessly!
        local_feat = ROOT / 'data/features'
        if local_feat.exists() and not local_feat.is_symlink():
            shutil.rmtree(local_feat)
        if not local_feat.exists():
            local_feat.symlink_to(FEATURES_DIR)
        print(f'Google Drive mounted and linked: {FEATURES_DIR}')
    except Exception as e:
        print(f'Google Drive not mounted ({e}). Using Colab local SSD storage.')
        FEATURES_DIR = ROOT / 'data/features'
        BACKUP_DIR = ROOT / 'results'
else:
    FEATURES_DIR = ROOT / 'data/features'
    BACKUP_DIR = ROOT / 'results'

FEATURES_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
print(f'Features storage directory: {FEATURES_DIR}')


## 3. Download Real PANDA ResNet50 Features to Google Drive
We download pre-extracted 1024-D ResNet50 patch embeddings for our 420 slides (300 train, 60 val, 60 test) from the Hugging Face computational pathology repository.
* Total download: ~600 MB (takes ~1-2 min on Colab's cloud connection).
* Once downloaded to Google Drive, future runs will find them cached instantly!


In [ ]:
# Download real features directly to Google Drive
MANIFEST = ROOT / 'data/common_split.csv'
existing = list(FEATURES_DIR.glob('*.pt'))
print(f'Status: {len(existing)} / 420 feature files currently in Google Drive.')

if len(existing) < 420:
    print('Downloading real ResNet50 features directly into Google Drive...')
    !python scripts/download_real_features.py --manifest {MANIFEST} --out-dir "{FEATURES_DIR}" --workers 16
else:
    print('All 420 real slide features already present in Google Drive!')


## 4. Train ABMIL Baseline (3 Seeds: 0, 1, 2)
We train across seeds 0, 1, 2 on the locked `data/common_split.csv` using Colab's GPU.
Each epoch will stream progress below. On a T4 GPU, training all 3 seeds takes ~3-4 minutes.


In [ ]:
CONFIG = ROOT / 'configs/abmil.yaml'
OUT = ROOT / 'results/metrics/abmil'
OUT.mkdir(parents=True, exist_ok=True)

!python -m src.train --model abmil --manifest {MANIFEST} --features-root "{FEATURES_DIR}" --config {CONFIG} --out-dir {OUT} --seeds 0 1 2


## 5. Baseline Results & Confusion Matrix
Display the official baseline metrics and the test-set confusion matrix.


In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

table_file = OUT / 'table_abmil.md'
if table_file.exists():
    print('### Official Baseline Results across 3 Seeds:')
    print(table_file.read_text())

agg_file = OUT / 'aggregate_abmil.json'
if agg_file.exists():
    agg = json.loads(agg_file.read_text())
    print('Per-seed summary:')
    display(pd.DataFrame(agg.get('per_run', [])))

conf_pngs = sorted(OUT.glob('confusion_*.png'))
if conf_pngs:
    print(f'Test Confusion Matrix ({conf_pngs[0].name}):')
    display(Image(filename=str(conf_pngs[0]), width=450))


## 6. Attention Analysis & Top-Patch Attribution
Person 1 sanity check: inspect attention weight distributions across test slides to verify the model is concentrating attention on informative patches.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.dataset import FeatureBagDataset, read_manifest, split_records
from src.evaluate_clam_sb import load_checkpoint
from src.heatmaps import attention_entropy, normalised_attention_entropy

ckpt_path = OUT / 'checkpoints/abmil_seed0_best.pt'
if ckpt_path.exists():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model, _ = load_checkpoint(ckpt_path, device=device)

    records = split_records(read_manifest(MANIFEST), 'test')
    dataset = FeatureBagDataset(records[:5], FEATURES_DIR, load_coords=False, train=False)

    fig, axes = plt.subplots(1, min(5, len(dataset)), figsize=(18, 3.5))
    if len(dataset) == 1:
        axes = [axes]

    print('Attention entropy on test slides (Seed 0):')
    for i in range(min(5, len(dataset))):
        item = dataset[i]
        rec = dataset.records[i]
        feats = item['features'].to(device).unsqueeze(0)
        with torch.no_grad():
            out = model(feats)
        attn = out['attention'].squeeze(0).cpu().numpy()
        pred = int(out['logits'].argmax(dim=-1).squeeze(0).cpu())

        ent = attention_entropy(attn)
        norm_ent = normalised_attention_entropy(attn)
        print(f'Slide {rec.image_id[:10]}... | True ISUP: {rec.label} | Pred ISUP: {pred} | Patches: {len(attn)} | Entropy: {ent:.3f}')

        ax = axes[i]
        ax.hist(attn, bins=30, color='#4c72b0', edgecolor='white')
        ax.set_title(f'True: {rec.label} | Pred: {pred}\n{rec.image_id[:8]}...', fontsize=10)
        ax.set_xlabel('Attention')
        if i == 0:
            ax.set_ylabel('Patch count')

    plt.tight_layout()
    plt.show()
else:
    print(f'Checkpoint {ckpt_path} not found. Run training cell first.')


## 7. Cloud Backup & Handoff to Person 2 (Archit)
Back up results to Google Drive and review the exact deliverables Person 2 needs for the paired statistical test.


In [ ]:
import shutil

# Backup results to Google Drive
target_backup = BACKUP_DIR / 'abmil'
if target_backup.exists():
    shutil.rmtree(target_backup)
shutil.copytree(OUT, target_backup)
print(f'Results permanently backed up to Google Drive at: {target_backup}')

# Deliverables for Person 2:
pred_file = OUT / 'predictions_abmil_seed0.csv'
print(f'Deliverable for Person 2: {pred_file.name} (exists: {pred_file.exists()})')
print('Archit can now run paired McNemar & Bootstrap tests with CLAM-SB using this file!')
